In [ ]:
# @title 1) Student Info & Config
STUDENT_NAME = "Воропаев Фёдор Игоревич"  # @param {type:"string"}
GROUP = "11-301"         # @param {type:"string"}
ASSIGMENT = "HW3_AUGMENTATION"
SEED = 42
START_DATE = "2026-03-10"
DUE_DATE = "2026-03-17"


# HW3 — Augmentations & Generalization (Albumentations)

**Topic:** Augmentations in real Computer Vision: how to make models robust without breaking labels.

**You will implement:**
1) `train_aug_moderate()` — a realistic (“moderate”) augmentation pipeline  
2) `train_aug_aggressive()` — a stronger (“aggressive”) pipeline (stress-test)  
3) `eval_preproc()` — deterministic preprocessing for evaluation  
4) consistent application to **image + mask + bounding boxes**  
5) a tiny **ablation**: baseline vs +augmentations on a synthetic domain-shift setup

**Rules**
- Don’t edit autograder/test cells.
- Run `Runtime → Restart and run all` before submitting.

Scoring: **100 points** (see tasks below).


In [ ]:
# Install deps (Colab-friendly)
!pip -q install albumentations==1.4.18 opencv-python-headless matplotlib pillow


In [ ]:
import random
import math
from dataclasses import dataclass
from datetime import datetime, timezone
from pathlib import Path
from typing import List, Tuple, Dict, Any, Optional

import numpy as np
import cv2
import matplotlib.pyplot as plt
import albumentations as A

plt.rcParams["figure.dpi"] = 140

# ---------- Reproducibility ----------
def seed_everything(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    try:
        A.set_seed(seed)
    except Exception:
        pass

seed_everything(SEED)

# ---------- Dates for penalty logic ----------
def _parse_date(s: str):
    # expects YYYY-MM-DD
    try:
        return datetime.strptime(s, "%Y-%m-%d").replace(tzinfo=timezone.utc)
    except Exception:
        return None

def _sec(td) -> float:
    return float(td.total_seconds())

start_dt = _parse_date(START_DATE)
due_dt = _parse_date(DUE_DATE)
submission_dt = datetime.now(timezone.utc)

# ---------- Display helpers ----------
def show_images(images, titles=None, cols=4, figsize=(12, 7)):
    n = len(images)
    cols = min(cols, n)
    rows = (n + cols - 1) // cols
    plt.figure(figsize=figsize)
    for i, img in enumerate(images):
        ax = plt.subplot(rows, cols, i + 1)
        ax.axis("off")
        if titles and i < len(titles):
            ax.set_title(titles[i], fontsize=9)
        if img.ndim == 2:
            plt.imshow(img, cmap="gray")
        else:
            plt.imshow(img)
    plt.tight_layout()
    plt.show()

def draw_bboxes(img_rgb, bboxes_xyxy, color=(0, 255, 0), thickness=2):
    out = img_rgb.copy()
    for (x1, y1, x2, y2) in bboxes_xyxy:
        cv2.rectangle(out, (int(x1), int(y1)), (int(x2), int(y2)), color, thickness)
    return out

def overlay_mask(img_rgb, mask01, alpha=0.35):
    overlay = img_rgb.copy()
    colored = np.zeros_like(img_rgb)
    colored[..., 0] = 255
    overlay = np.where(mask01[..., None].astype(bool), (1-alpha)*overlay + alpha*colored, overlay)
    return overlay.astype(np.uint8)

# ---------- Scoring ----------
SCORES: Dict[str, float] = {}
def _set_score(task: str, pts: float):
    SCORES[task] = float(pts)

def _total_score():
    return float(sum(SCORES.values()))

def _print_scores():
    print("Scores:")
    for k in sorted(SCORES.keys()):
        print(f"  {k}: {SCORES[k]:.1f}")
    print("TOTAL:", _total_score())


## Tasks

- **Task 1 (20 pts)** — implement `train_aug_moderate()`  
- **Task 2 (20 pts)** — implement `train_aug_aggressive()`  
- **Task 3 (10 pts)** — implement `eval_preproc()` (no randomness)  
- **Task 4 (20 pts)** — apply augmentations consistently to **image + mask + bboxes**  
- **Task 5 (10 pts)** — reproducibility: reseeding should reproduce the same augmented sample  
- **Task 6 (20 pts)** — mini ablation: baseline vs +moderate aug under domain shift

> You are graded by tests below each task.


In [ ]:
# Synthetic scene (image + mask + two bboxes) for tasks 4-5
def make_synthetic_scene(h=256, w=256):
    img = np.zeros((h, w, 3), dtype=np.uint8)

    # background gradient
    for y in range(h):
        img[y, :, 0] = np.clip(20 + y * 0.4, 0, 255)
        img[y, :, 1] = np.clip(30 + y * 0.3, 0, 255)
        img[y, :, 2] = np.clip(50 + y * 0.2, 0, 255)

    # mask: "person-like" blob
    mask = np.zeros((h, w), dtype=np.uint8)
    center = (int(w*0.36), int(h*0.60))
    cv2.ellipse(mask, center, (30, 55), 0, 0, 360, 1, -1)
    cv2.circle(mask, (center[0], int(h*0.40)), 18, 1, -1)
    img[mask.astype(bool)] = (60, 200, 80)  # BGR green-ish

    # boxes: two rectangles
    bboxes = []
    x1, y1, x2, y2 = int(w*0.62), int(h*0.55), int(w*0.88), int(h*0.82)
    cv2.rectangle(img, (x1, y1), (x2, y2), (220, 120, 70), -1)
    bboxes.append([x1, y1, x2, y2])

    x1, y1, x2, y2 = int(w*0.08), int(h*0.12), int(w*0.28), int(h*0.28)
    cv2.rectangle(img, (x1, y1), (x2, y2), (60, 130, 220), -1)
    bboxes.append([x1, y1, x2, y2])

    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    return img_rgb, mask.astype(np.uint8), bboxes

img_rgb, mask01, bboxes = make_synthetic_scene()
vis = overlay_mask(draw_bboxes(img_rgb, bboxes), mask01)
show_images([vis], ["Base sample (image + bboxes + mask)"], cols=1, figsize=(6, 6))


In [ ]:
def _flatten_transforms(t) -> List[Any]:
    # Returns a flat list of transforms inside Compose/OneOf/SomeOf etc.
    out = []
    if t is None:
        return out
    if isinstance(t, A.Compose):
        for x in t.transforms:
            out.extend(_flatten_transforms(x))
        return out
    # OneOf / SomeOf
    if hasattr(t, "transforms") and isinstance(getattr(t, "transforms"), (list, tuple)):
        out.append(t)
        for x in t.transforms:
            out.extend(_flatten_transforms(x))
        return out
    out.append(t)
    return out

def _names_of(pipeline) -> List[str]:
    flat = _flatten_transforms(pipeline)
    return [x.__class__.__name__ for x in flat]

def _has_any(pipeline, names: List[str]) -> bool:
    s = set(_names_of(pipeline))
    return any(n in s for n in names)

def _assert(cond: bool, msg: str):
    if not cond:
        raise AssertionError(msg)


## Task 1 (20 pts) — `train_aug_moderate()`

Implement a **moderate** training augmentation pipeline.

**Requirements (checked by tests):**
- Output size must be **256×256**
- Must include at least:
  - one **flip** (`HorizontalFlip`)
  - one **small geometry** transform (one of: `ShiftScaleRotate`, `Affine`, `Rotate`)
  - one **photometric** transform (`RandomBrightnessContrast` or `HueSaturationValue` or `RandomGamma`)
  - one **noise/blur/artifact** transform (one of: `GaussNoise`, `GaussianBlur`, `MotionBlur`, `ImageCompression`, `Downscale`)
- Must be **random** (not deterministic)

Tip: keep it realistic: values that can happen in production.


In [ ]:
# TODO: implement
def train_aug_moderate() -> A.Compose:
    """Return Albumentations Compose for training (moderate profile)."""
    # YOUR CODE HERE
    raise NotImplementedError()


In [ ]:
# Tests for Task 1
try:
    aug = train_aug_moderate()
    _assert(isinstance(aug, A.Compose), "train_aug_moderate() must return A.Compose")
    # output size
    x = aug(image=img_rgb)["image"]
    _assert(tuple(x.shape[:2]) == (256, 256), f"Expected (256,256) output, got {x.shape[:2]}")
    # required categories
    _assert(_has_any(aug, ["HorizontalFlip"]), "Moderate aug must include HorizontalFlip")
    _assert(_has_any(aug, ["ShiftScaleRotate", "Affine", "Rotate"]), "Moderate aug must include one of: ShiftScaleRotate/Affine/Rotate")
    _assert(_has_any(aug, ["RandomBrightnessContrast", "HueSaturationValue", "RandomGamma"]), "Moderate aug must include photometric transform")
    _assert(_has_any(aug, ["GaussNoise", "GaussianBlur", "MotionBlur", "ImageCompression", "Downscale"]), "Moderate aug must include noise/blur/artifact transform")
    # randomness check
    seed_everything(SEED)
    a1 = aug(image=img_rgb)["image"]
    seed_everything(SEED)
    a2 = aug(image=img_rgb)["image"]
    # with same seed we expect same output (reproducible)
    _assert(np.array_equal(a1, a2), "With reseeding to the same SEED, moderate aug should reproduce identical output")
    # without reseeding we expect usually different
    b1 = aug(image=img_rgb)["image"]
    b2 = aug(image=img_rgb)["image"]
    _assert(not np.array_equal(b1, b2), "Moderate aug seems deterministic; add randomness (p<1 or random params)")
    _set_score("task1", 20)
    print("✅ Task 1 passed")
except Exception as e:
    print("❌ Task 1 failed:", e)
    _set_score("task1", 0)


## Task 2 (20 pts) — `train_aug_aggressive()`

Implement an **aggressive** training augmentation pipeline (stress-test).

**Requirements (checked by tests):**
- Output size must be **256×256**
- Must include at least:
  - one **strong geometry** transform (one of: `Perspective`, `GridDistortion`, `OpticalDistortion`, `ElasticTransform`)
  - one **occlusion** transform (`CoarseDropout` or `GridDropout`)
  - one **artifact** transform (`ImageCompression` or `Downscale`)
- Must be random and reproducible with reseeding.

Tip: aggressive ≠ nonsense. It should still look like your production world.


In [ ]:
# TODO: implement
def train_aug_aggressive() -> A.Compose:
    """Return Albumentations Compose for training (aggressive profile)."""
    # YOUR CODE HERE
    raise NotImplementedError()


In [ ]:
# Tests for Task 2
try:
    aug = train_aug_aggressive()
    _assert(isinstance(aug, A.Compose), "train_aug_aggressive() must return A.Compose")
    x = aug(image=img_rgb)["image"]
    _assert(tuple(x.shape[:2]) == (256, 256), f"Expected (256,256) output, got {x.shape[:2]}")
    _assert(_has_any(aug, ["Perspective", "GridDistortion", "OpticalDistortion", "ElasticTransform"]), "Aggressive aug must include strong geometry transform")
    _assert(_has_any(aug, ["CoarseDropout", "GridDropout"]), "Aggressive aug must include occlusion (CoarseDropout or GridDropout)")
    _assert(_has_any(aug, ["ImageCompression", "Downscale"]), "Aggressive aug must include artifact (ImageCompression or Downscale)")
    seed_everything(SEED)
    a1 = aug(image=img_rgb)["image"]
    seed_everything(SEED)
    a2 = aug(image=img_rgb)["image"]
    _assert(np.array_equal(a1, a2), "With reseeding to the same SEED, aggressive aug should reproduce identical output")
    b1 = aug(image=img_rgb)["image"]
    b2 = aug(image=img_rgb)["image"]
    _assert(not np.array_equal(b1, b2), "Aggressive aug seems deterministic; add randomness")
    _set_score("task2", 20)
    print("✅ Task 2 passed")
except Exception as e:
    print("❌ Task 2 failed:", e)
    _set_score("task2", 0)


## Task 3 (10 pts) — `eval_preproc()`

Implement deterministic preprocessing for evaluation/inference.

**Requirements (checked by tests):**
- Output size must be **256×256**
- Must NOT include random transforms (no flips, no random crops, etc.)
- Recommended: `LongestMaxSize` + `PadIfNeeded` + (optional) `Normalize`


In [ ]:
# TODO: implement
def eval_preproc() -> A.Compose:
    """Return deterministic preprocessing Compose for eval/test."""
    # YOUR CODE HERE
    raise NotImplementedError()


In [ ]:
# Tests for Task 3
try:
    pp = eval_preproc()
    _assert(isinstance(pp, A.Compose), "eval_preproc() must return A.Compose")
    x1 = pp(image=img_rgb)["image"]
    x2 = pp(image=img_rgb)["image"]
    _assert(tuple(x1.shape[:2]) == (256, 256), f"Expected (256,256) output, got {x1.shape[:2]}")
    _assert(np.array_equal(x1, x2), "Eval preprocessing must be deterministic (same input => same output)")
    # Should not contain typical random transforms
    forbidden = {"HorizontalFlip","VerticalFlip","RandomRotate90","RandomCrop","RandomResizedCrop","ShiftScaleRotate","Rotate","Affine","CoarseDropout","GridDropout"}
    present = set(_names_of(pp))
    _assert(len(forbidden.intersection(present)) == 0, f"Eval preprocessing contains random transforms: {forbidden.intersection(present)}")
    _set_score("task3", 10)
    print("✅ Task 3 passed")
except Exception as e:
    print("❌ Task 3 failed:", e)
    _set_score("task3", 0)


## Task 4 (20 pts) — Apply augmentations to image + mask + bounding boxes

Implement `apply_aug_to_sample(pipeline, image, mask, bboxes)`.

**Requirements (checked by tests):**
- Uses Albumentations `Compose(..., bbox_params=...)` with `pascal_voc` format
- Applies transforms **consistently** to:
  - `image` (RGB)
  - `mask` (0/1)
  - `bboxes` in **xyxy**
- Output must satisfy:
  - `image.shape == (256,256,3)`
  - mask is **binary** (only 0/1)
  - bboxes are inside image bounds (clipped OK)


In [ ]:
# TODO: implement
def apply_aug_to_sample(pipeline: A.Compose, image_rgb: np.ndarray, mask01: np.ndarray, bboxes_xyxy: List[List[float]]):
    """Return augmented (image, mask, bboxes)."""
    # YOUR CODE HERE
    raise NotImplementedError()


In [ ]:
# Tests for Task 4
try:
    aug = train_aug_moderate()
    seed_everything(SEED)
    out_img, out_mask, out_boxes = apply_aug_to_sample(aug, img_rgb, mask01, bboxes)

    _assert(out_img.shape == (256,256,3), f"Expected image shape (256,256,3), got {out_img.shape}")
    _assert(out_mask.shape == (256,256), f"Expected mask shape (256,256), got {out_mask.shape}")
    uniq = set(np.unique(out_mask).tolist())
    _assert(uniq.issubset({0,1}), f"Mask must be binary (0/1), got values: {sorted(list(uniq))}")

    # bbox bounds
    for (x1,y1,x2,y2) in out_boxes:
        _assert(0 <= x1 <= 256 and 0 <= x2 <= 256 and 0 <= y1 <= 256 and 0 <= y2 <= 256, "BBoxes must be within [0,256] after aug")
        _assert(x2 >= x1 and y2 >= y1, "Invalid bbox (x2<x1 or y2<y1)")

    # visualize a sample
    vis = overlay_mask(draw_bboxes(out_img, out_boxes), out_mask)
    show_images([vis], ["Augmented sample (moderate)"], cols=1, figsize=(6,6))

    _set_score("task4", 20)
    print("✅ Task 4 passed")
except Exception as e:
    print("❌ Task 4 failed:", e)
    _set_score("task4", 0)


## Task 5 (10 pts) — Reproducibility by reseeding

Implement `sample_two_times_same_seed(pipeline)`:
- reseed with `SEED`
- apply augmentation twice
- return both images

**Goal:** show that with reseeding you can reproduce the same augmented sample (important for debugging).


In [ ]:
# TODO: implement
def sample_two_times_same_seed(pipeline: A.Compose) -> Tuple[np.ndarray, np.ndarray]:
    # YOUR CODE HERE
    raise NotImplementedError()


In [ ]:
# Tests for Task 5
try:
    aug = train_aug_aggressive()
    a1, a2 = sample_two_times_same_seed(aug)
    _assert(a1.shape == (256,256,3) and a2.shape == (256,256,3), "Returned images must be (256,256,3)")
    _assert(np.array_equal(a1, a2), "With reseeding, two samples must be identical")
    _set_score("task5", 10)
    print("✅ Task 5 passed")
except Exception as e:
    print("❌ Task 5 failed:", e)
    _set_score("task5", 0)


## Task 6 (20 pts) — Mini ablation: baseline vs +moderate aug (domain shift)

We build a synthetic classification dataset with a **spurious correlation**:
- Label depends on **shape** (circle vs rectangle)
- In TRAIN, background brightness correlates with the label (easy shortcut)
- In DEV, the correlation is **broken** (background swapped/random)

Baseline (no aug) often learns the shortcut and fails on DEV.
With **moderate photometric + geometry** augmentations, the model should rely more on shape.

You must implement:
- `run_ablation()` that trains:
  1) baseline (no aug)
  2) augmented (using `train_aug_moderate()`)
and returns a dict with keys:
- `baseline_dev_acc`
- `augmented_dev_acc`
- `delta`

**Passing condition (checked by tests):**
- `delta >= 0.15` and `augmented_dev_acc >= 0.75`

Keep epochs small (already set). Runtime should stay under a couple minutes.


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("DEVICE:", DEVICE)

TARGET = 64  # smaller for speed

def _draw_shape_sample(label: int, domain: str, rng: np.random.RandomState) -> np.ndarray:
    # label: 0=circle, 1=rectangle
    h = w = TARGET
    img = np.zeros((h, w, 3), dtype=np.uint8)

    # background brightness: spurious correlation in TRAIN
    if domain == "train":
        bg = 210 if label == 0 else 50
    else:
        # break the correlation in dev
        bg = 50 if label == 0 else 210

    img[:] = (bg, bg, bg)

    # draw shape with fixed mid-gray to avoid color leakage
    color = (120, 120, 120)
    cx, cy = int(rng.uniform(20, 44)), int(rng.uniform(20, 44))
    size = int(rng.uniform(10, 16))
    if label == 0:
        cv2.circle(img, (cx, cy), size, color, -1)
    else:
        cv2.rectangle(img, (cx-size, cy-size), (cx+size, cy+size), color, -1)

    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    return img_rgb

class ShapesDataset(Dataset):
    def __init__(self, n: int, domain: str, aug: Optional[A.Compose], seed: int):
        self.n = n
        self.domain = domain
        self.aug = aug
        self.rng = np.random.RandomState(seed)
        # pre-generate labels for determinism
        self.labels = self.rng.randint(0, 2, size=n).astype(int)

    def __len__(self):
        return self.n

    def __getitem__(self, idx):
        label = int(self.labels[idx])
        img = _draw_shape_sample(label, self.domain, self.rng)

        if self.aug is not None:
            img = self.aug(image=img)["image"]

        # to tensor
        x = torch.from_numpy(img).float().permute(2,0,1) / 255.0
        y = torch.tensor(label, dtype=torch.long)
        return x, y

class TinyCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(3, 16, 3, padding=1)
        self.conv2 = nn.Conv2d(16, 32, 3, padding=1)
        self.conv3 = nn.Conv2d(32, 64, 3, padding=1)
        self.pool = nn.MaxPool2d(2)
        self.fc = nn.Linear(64*(TARGET//8)*(TARGET//8), 2)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = self.pool(F.relu(self.conv3(x)))
        x = x.flatten(1)
        return self.fc(x)

@torch.no_grad()
def eval_acc(model, loader):
    model.eval()
    correct = 0
    total = 0
    for x, y in loader:
        x, y = x.to(DEVICE), y.to(DEVICE)
        logits = model(x)
        pred = logits.argmax(1)
        correct += int((pred == y).sum().item())
        total += int(y.numel())
    return correct / max(1, total)

def train_one(model, train_loader, epochs=3, lr=1e-3):
    model.train()
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    for _ in range(epochs):
        for x, y in train_loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            opt.zero_grad()
            logits = model(x)
            loss = F.cross_entropy(logits, y)
            loss.backward()
            opt.step()


In [ ]:
# TODO: implement
def run_ablation() -> Dict[str, float]:
    """Train baseline vs augmented and return metrics dict."""
    # YOUR CODE HERE
    raise NotImplementedError()


In [ ]:
# Tests for Task 6
try:
    seed_everything(SEED)
    torch.manual_seed(SEED)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(SEED)

    res = run_ablation()
    _assert(isinstance(res, dict), "run_ablation() must return a dict")
    for k in ["baseline_dev_acc", "augmented_dev_acc", "delta"]:
        _assert(k in res, f"Missing key: {k}")

    b = float(res["baseline_dev_acc"])
    a = float(res["augmented_dev_acc"])
    d = float(res["delta"])

    print("baseline_dev_acc:", b)
    print("augmented_dev_acc:", a)
    print("delta:", d)

    _assert(abs(d - (a - b)) < 1e-6, "delta must equal augmented_dev_acc - baseline_dev_acc")
    _assert(d >= 0.15, "Expected delta >= 0.15 (augmentations should help under domain shift)")
    _assert(a >= 0.75, "Expected augmented_dev_acc >= 0.75")

    _set_score("task6", 20)
    print("✅ Task 6 passed")
except Exception as e:
    print("❌ Task 6 failed:", e)
    _set_score("task6", 0)


In [ ]:
# FINAL: compute total points (0..100)
total = _total_score()
_print_scores()
print("\nTOTAL POINTS / 100 =", total)


In [ ]:
def penalty_fraction(start_dt, due_dt, now_dt) -> float:
    if not (start_dt and due_dt and now_dt):
        return 0.0
    window = _sec(due_dt - start_dt)
    if window <= 0:
        return 1.0 if now_dt > due_dt else 0.0
    late = max(0.0, _sec(now_dt - due_dt))
    return min(1.0, late / window)

# применяем штраф
try:
    pf = penalty_fraction(start_dt, due_dt, submission_dt)
except NameError:
    from datetime import timezone
    pf = 0.0
final_score = max(0.0, total * (1.0 - min(1.0, pf)))
print(final_score)
import json
final = {
    "name": STUDENT_NAME,
    "group": GROUP,
    "assignment":ASSIGMENT,
    "score": float(total),
    "penalty_score": float(final_score),
    "due_date": DUE_DATE,
    "start_date": START_DATE,
}



In [ ]:
print(json.dumps(final, ensure_ascii=False))